# Lesson 07 - योजना डिज़ाइन पैटर्न

यह नोटबुक Microsoft Agent Framework का उपयोग करते हुए AI एजेंट्स के लिए **योजना डिज़ाइन पैटर्न** को प्रदर्शित करता है।
आप सीखेंगे कि जटिल यात्रा अनुरोधों को संरचित उपकार्य में कैसे विभाजित किया जाए, उन्हें विशेषज्ञ एजेंट्स को सौंपा जाए,
और परिणामी योजना को निष्पादित किया जाए — वह सब पायडैंटिक मॉडलों द्वारा संचालित संरचित आउटपुट का उपयोग करके।


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## कार्य विखंडन

कार्य विखंडन योजना डिजाइन पैटर्न का मूल है। एक ही एजेंट से एक जटिल अनुरोध को शुरू से अंत तक संभालने के बजाय, हम समस्या को छोटे, स्पष्ट परिभाषित **उपकार्य** में विभाजित करते हैं। प्रत्येक उपकार्य को एक विशेषज्ञ एजेंट (जैसे, उड़ानें, होटल, गतिविधियाँ) को स्पष्ट प्राथमिकताओं और निर्भरता क्रम के साथ सौंपा जाता है।

यह दृष्टिकोण कई लाभ प्रदान करता है:
- **स्पष्टता**: प्रत्येक उपकार्य की एकल जिम्मेदारी होती है।
- **समानांतरता**: स्वतंत्र उपकार्य एक साथ चल सकते हैं।
- **विश्वसनीयता**: विफलताएं व्यक्तिगत उपकार्य तक सीमित रहती हैं।
- **बजट ट्रैकिंग**: लागतों का अनुमान प्रत्येक उपकार्य के लिए लगाया जाता है और कुल में जोड़ा जाता है।


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## संरचित आउटपुट के साथ एक योजना एजेंट बनाना

योजनाकार एजेंट एक **फ्रंट डेस्क समन्वयक** के रूप में कार्य करता है। एक उच्च-स्तरीय यात्रा अनुरोध दिए जाने पर यह एक संरचित `TravelPlan` प्रदान करता है — अनुरोध को उपकार्य में विभाजित करते हुए, प्राथमिकताएं सेट करता है, और निर्भरताएं पहचानता है ताकि एक कंसीयर्ज़ या निष्पादन परत कार्य को पूरा कर सके।


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## विशेषज्ञ उपकरणों के साथ एक योजना को क्रियान्वित करना

एक बार फ्रंट डेस्क एजेंट ने एक संरचित योजना तैयार कर ली, तो **कंसीयर्ज एजेंट** उसे क्रियान्वित करता है। प्रत्येक विशेषज्ञ उपकरण एक उपकार्य की एक श्रेणी (उड़ानें, होटल, गतिविधियाँ) को संभालता है। कंसीयर्ज योजना के उपकार्यों को निर्भरता क्रम में दोहराता है और प्रत्येक को उपयुक्त उपकरण को सौंपता है।


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## सारांश

इस पाठ में आपने AI एजेंट्स के लिए **योजना डिजाइन पैटर्न** सीखा:

1. **कार्य विभाजन** — एक फ्रंट डेस्क योजना एजेंट जटिल यात्रा अनुरोध को संरचित उपकार्य में तोड़ता है, पायडांटिक मॉडल का उपयोग करके, प्रत्येक को विशेषज्ञ एजेंट को प्राथमिकताएं और निर्भरता के साथ सौंपता है।
2. **संरचित आउटपुट** — `response_format` पास करके एजेंट एक मान्य `TravelPlan` ऑब्जेक्ट लौटाता है न कि मुक्त रूप का टेक्स्ट, जिससे डाउनस्ट्रीम प्रोसेसिंग विश्वसनीय हो जाती है।
3. **योजना निष्पादन** — एक कंसीयर्ज एजेंट विशेषज्ञ उपकरणों (`book_flight`, `reserve_hotel`, `book_activity`) का उपयोग करते हुए उपकार्यों पर काम करता है, योजना को पूरा करता है और परिणाम रिपोर्ट करता है।

यह पैटर्न *क्या करना है* (योजना) को *कैसे करना है* (निष्पादन) से अलग करता है, जिससे एजेंट अधिक मॉड्यूलर, परीक्षात्मक और विस्तारित करने में आसान हो जाते हैं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या असंगतियाँ हो सकती हैं। मूल दस्तावेज़, जो इसकी मूल भाषा में है, को प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या भ्रांतियों के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
